# Mobile Price Predictions with MLflow Tracking

In this notebook, you will learn how to build a Kubeflow pipeline using lightweight components, enhanced with **MLflow tracking and model registry** integration. If you are looking for an example without MLflow, see the [Mobile Price Predictions](../../pipelines/lightweight-components/mobile-price-classifications.ipynb) notebook.

For this tutorial, we will utilize the Mobile Price Classification dataset available [on Kaggle](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification?datasetId=11167&sortBy=voteCount).

## MLflow Integration Highlights

- The `train_model` component creates an **MLflow experiment** per user, logs hyperparameters, and **registers the model** in the MLflow Model Registry.
- The `evaluate_model` component **resumes the same MLflow run** and logs validation metrics and a confusion matrix image.
- MLflow credentials are injected securely into pipeline tasks via a **Kubernetes secret** (`mlflow-credentials`).

In [ ]:
%pip -q install kagglehub

In [ ]:
# Load the MLflow credential helpers without starting interactive setup.
%run -n ~/examples/src/pk_helpers/mlflow_credentials.py

In [ ]:
import os
import pandas as pd
import numpy as np
import kfp
import kfp.dsl as dsl
from kfp.dsl import HTML, Input, Output, Dataset, Artifact, Model, ClassificationMetrics, Markdown
from kfp.client import Client
from typing import Dict, Tuple, List
from kfp import kubernetes
from kfp.kubernetes import use_secret_as_env
import s3fs

## MLflow Setup

Pipeline components run in separate pods, so they read MLflow credentials from
the `mlflow-credentials` Kubernetes secret. First, [create a Personal Access Token
in MLflow](/mlflow/oidc/ui/auth) and copy it. Then run the credential setup cell
below. The script prompts for your MLflow tracking URI, email address, and token,
then creates or updates the `mlflow-credentials` Kubernetes Secret for you.

If the Secret already exists, the script asks before replacing it and leaves it
unchanged by default. The separate `%run -n` cell only loads the helper functions
used later; it does not prompt or modify the Secret. Automated tests skip the
interactive setup cell and use the preconfigured Secret validated during preflight.

In [ ]:
# Prompts for your MLflow URI, email, and PAT, then creates or updates the Secret.
setup_mlflow_credentials()

In [ ]:
# Check the credentials before submitting the pipeline so a missing secret
# produces a clear error here rather than inside a pipeline component.
require_mlflow_secret()

In [ ]:
def add_env_vars_to_tasks(task_list: list) -> None:
    """Inject MLflow credentials from the Kubernetes secret into pipeline tasks."""
    for task in task_list:
        use_secret_as_env(
            task,
            secret_name="mlflow-credentials",
            secret_key_to_env={
                "MLFLOW_TRACKING_URI": "MLFLOW_TRACKING_URI",
                "MLFLOW_TRACKING_USERNAME": "MLFLOW_TRACKING_USERNAME",
                "MLFLOW_TRACKING_PASSWORD": "MLFLOW_TRACKING_PASSWORD",
            }
        )

In [ ]:
# Configuration for object storage bucket
# Uses the default prokube bucket for this namespace: <namespace>-data
# You can check your buckets by opening your object storage browser or using the configured storage CLI alias.
with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace", "r") as namespace_file:
    namespace = namespace_file.read().strip()
s3_bucket = f"{namespace}-data"
s3_dataset_path = f"{s3_bucket}/mobile-price-classification"

## Download Dataset From Kagglehub and Upload to Object Storage with Python

In [ ]:
# Download Dataset From Kagglehub
import kagglehub
dataset_path = kagglehub.dataset_download("iabhishekofficial/mobile-price-classification")
print(f"Kaggle dataset downloaded to: {dataset_path}")

In [ ]:
# Upload dataset to object storage
if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
    raise ValueError("AWS credentials not found in environment variables.")

# Initialize S3 filesystem
s3 = s3fs.S3FileSystem()

# Upload the dataset to object storage
s3.put(f"{dataset_path}/train.csv", f"{s3_dataset_path}/train.csv")
s3.put(f"{dataset_path}/test.csv", f"{s3_dataset_path}/test.csv")

print(s3.ls(s3_dataset_path))

## Create components

### Read data

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "pyarrow", "s3fs"],
    base_image="python:3.11",
)
def read_data(
    train_data_path: str,
    test_data_path: str,
    train_df: Output[Dataset],
    test_df: Output[Dataset],    
):
    """Reads training and test data writes it to pipeline artifacts as parquet."""
    import pandas as pd

    df_train = pd.read_csv(train_data_path)
    df_test = pd.read_csv(test_data_path)
    
    df_train.to_parquet(train_df.path)
    df_test.to_parquet(test_df.path)

### Split data

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "pyarrow"],
    base_image="python:3.11",
)
def split_data(
    train_df: Input[Dataset],
    x_train_df: Output[Dataset],
    y_train_df: Output[Dataset],
    x_val_df: Output[Dataset],
    y_val_df: Output[Dataset],
    test_size: float = 0.5,
    seed: int = 42,
):
    """Splits the provided dataset into training and validation sets."""
    
    import pandas as pd
    from sklearn.model_selection import train_test_split 

    # Read the input dataset
    data = pd.read_parquet(train_df.path)

    # Separate target from features
    y = data["price_range"].to_frame()
    x_data = data.drop(["price_range"], axis=1)
    
    # Split the data into training and validation sets
    x_train, x_val, y_train, y_val = train_test_split(x_data, y, test_size=test_size, random_state=seed)

    # Save the splitted data to their respective output paths
    for object, artifact in zip((x_train, x_val, y_train, y_val), (x_train_df, x_val_df, y_train_df, y_val_df)):
        object.to_parquet(artifact.path)


### Fit scaler

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "pyarrow"],
    base_image="python:3.11",
)
def fit_scaler(
    train_x: Input[Dataset],
    fitted_scaler: Output[Artifact]
):
    """
    Fits a MinMaxScaler on the provided training data and saves the fitted scaler.
    """
    from sklearn.preprocessing import MinMaxScaler
    from joblib import dump
    import pandas as pd
    
    # Read the input dataset
    x_train = pd.read_parquet(train_x.path)
    
    # Fit the MinMaxScaler on the training data
    scaler = MinMaxScaler()
    scaler.fit(x_train)
    
    # Save the fitted scaler
    dump(scaler, fitted_scaler.path)

### Run grid search

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "pyarrow"],
    base_image="python:3.11",
)
def tune_hyperparams(
    train_x: Input[Dataset],
    train_y: Input[Dataset],
    fitted_scaler: Input[Artifact],
    C: List = [1, 0.1, 0.25, 0.5, 2, 0.75],
    kernel: List = ["linear", "rbf"],
    gamma: List = ["auto", 0.01, 0.001, 0.0001, 1],
    decision_function_shape: List[str] = ["ovo", "ovr"],
    seed: int = 42,
) -> dict:
    """
    Performs hyperparameter tuning using GridSearchCV for a SVM classifier on the provided training data.
    Returns the best hyperparameters found.
    """
    import pandas as pd
    from sklearn.model_selection import GridSearchCV
    from sklearn.svm import SVC
    from joblib import load

    # Load the fitted scaler
    scaler = load(fitted_scaler.path)

    # Read and preprocess the training data
    x_train, y_train = [pd.read_parquet(path) for path in (train_x.path, train_y.path)]
    x_train = pd.DataFrame(scaler.transform(x_train), columns=x_train.columns)

    # Initialize SVM with a random seed
    svm = SVC(random_state=seed)

    # Define grid search with provided hyperparameters
    grid_svm = GridSearchCV(
        estimator=svm,
        cv=5,
        param_grid=dict(
            kernel=kernel, 
            C=C, 
            gamma=gamma, 
            decision_function_shape=decision_function_shape
        )
    )

    # Perform grid search
    grid_svm.fit(x_train, y_train['price_range'].values)
    
    # Print the best score found
    print("Best score: ", grid_svm.best_score_)

    # Return the best hyperparameters
    return grid_svm.best_params_

### Train model with optimal hyper parameters

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "pyarrow", "mlflow==3.10.0", "s3fs"],
    base_image="python:3.11",
)
def train_model(
    train_x: Input[Dataset],
    train_y: Input[Dataset],
    fitted_scaler: Input[Artifact],
    hparams: Dict,
    trained_model: Output[Model],
    seed: int = 42,
) -> dict:
    """
    Trains an SVM classifier on the provided training data using the best hyperparameters from tuning.
    The trained model is saved as a KFP artifact and logged to MLflow with model registration.
    Returns the MLflow run info as a dict for downstream components.
    """
    import os

    import mlflow
    import pandas as pd
    from mlflow.models import infer_signature
    from sklearn.svm import SVC
    from joblib import dump, load

    # Load the fitted scaler
    scaler = load(fitted_scaler.path)

    # Read and preprocess the training data
    x_train, y_train = [pd.read_parquet(path) for path in (train_x.path, train_y.path)]
    x_train = pd.DataFrame(scaler.transform(x_train), columns=x_train.columns)

    # Initialize SVM with the best hyperparameters and a random seed
    svm_model = SVC(random_state=seed, **hparams)

    # Train the SVM model
    svm_model.fit(x_train, y_train['price_range'].values)

    # Save the trained model as KFP artifact (keeps existing pipeline flow intact)
    dump(svm_model, trained_model.path)

    # --- MLflow tracking ---
    username = os.getenv('MLFLOW_TRACKING_USERNAME', 'unknown').split('@')[0]
    mlflow.set_experiment(f'Mobile Price Classification {username}')

    with mlflow.start_run() as run:
        # Log hyperparameters
        mlflow.log_params(hparams)
        mlflow.log_param('seed', seed)

        mlflow.set_tag('Training Info', 'SVM model for mobile price classification')

        # Infer model signature and log model to registry
        signature = infer_signature(x_train, svm_model.predict(x_train))
        mlflow.sklearn.log_model(
            sk_model=svm_model,
            artifact_path='svm-model',
            signature=signature,
            input_example=x_train.head(5),
            registered_model_name=f'mobile-price-svm-{username}',
        )

    return run.to_dictionary()


### Evaluate model on validation dataset

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "pyarrow", "mlflow==3.10.0", "s3fs", "matplotlib"],
    base_image="python:3.11",
)
def evaluate_model(
    val_x: Input[Dataset],
    val_y: Input[Dataset],
    fitted_scaler: Input[Artifact],
    trained_model: Input[Model],
    mlflow_run: dict,
    confusion_matrix_plot: Output[ClassificationMetrics],
    classification_report_md: Output[Markdown]
):
    """
    Evaluates the performance of a trained SVM model using validation data.
    Outputs a confusion matrix plot and a markdown classification report.
    Also logs evaluation metrics and the confusion matrix to the MLflow run.
    """
    import mlflow
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from sklearn.svm import SVC
    from joblib import load
    from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

    # Load the fitted scaler and preprocess validation data
    scaler = load(fitted_scaler.path)
    x_val, y_val = [pd.read_parquet(path) for path in (val_x.path, val_y.path)]
    x_val = pd.DataFrame(scaler.transform(x_val), columns=x_val.columns)

    # Load the trained SVM model and make predictions on validation data
    svm_model = load(trained_model.path)
    predictions = svm_model.predict(x_val)

    # Log the confusion matrix (KFP artifact)
    confusion_matrix_plot.log_confusion_matrix(
        [str(v) for v in y_val['price_range'].unique()],
        confusion_matrix(y_val['price_range'].values.tolist(), predictions.tolist()).tolist()
    )

    # Create the markdown content for classification report
    report = classification_report(y_val['price_range'].values, predictions)
    markdown_content = f"```\n{report}\n```"

    # Write the content to a Markdown file
    with open(classification_report_md.path, 'w') as f:
        f.write(markdown_content)

    # --- MLflow tracking ---
    run_id = mlflow_run['info']['run_id']
    report_dict = classification_report(y_val['price_range'].values, predictions, output_dict=True)

    with mlflow.start_run(run_id=run_id):
        mlflow.log_metric('accuracy', report_dict['accuracy'])
        mlflow.log_metric('precision_weighted', report_dict['weighted avg']['precision'])
        mlflow.log_metric('recall_weighted', report_dict['weighted avg']['recall'])
        mlflow.log_metric('f1_weighted', report_dict['weighted avg']['f1-score'])

        # Log confusion matrix as an image artifact
        fig, ax = plt.subplots(figsize=(8, 6))
        cm = confusion_matrix(y_val['price_range'].values, predictions)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(ax=ax)
        ax.set_title('Confusion Matrix')
        mlflow.log_figure(fig, 'confusion_matrix.png')
        plt.close(fig)

### Run predictions on test dataset

In [ ]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "pyarrow", "plotly"],
    base_image="python:3.11",
)
def test_model(
    test_x: Input[Dataset],
    trained_model: Input[Model],
    fitted_scaler: Input[Artifact],
    column_x: str,
    column_y: str,
    scatter_plot: Output[HTML]
):
    """
    Test a trained SVM model on provided test data and produce a scatter plot.
    The scatter plot will have points colored by the predicted class based on two columns
    specified by the user.
    """
    import pandas as pd
    from joblib import load
    import plotly.express as px

    # Load the fitted scaler and preprocess test data
    scaler = load(fitted_scaler.path)
    x_test = pd.read_parquet(test_x.path)
    x_test = x_test.drop('id', axis=1)
    x_test = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns)

    # Load the trained SVM model and make predictions on test data
    svm_model = load(trained_model.path)
    predictions = svm_model.predict(x_test)

    # Add predictions as a column to the x_test DataFrame for visualization
    x_test['Predicted Class'] = predictions

    # Create the scatter plot using plotly
    fig = px.scatter(x_test,
                     x=column_x,
                     y=column_y,
                     color='Predicted Class',
                     color_continuous_scale='Viridis',
                     title=f"Scatter plot of {column_x} vs. {column_y} colored by Predicted Class",
                     template='plotly_dark')

    # Save the plot as an HTML file
    fig.write_html(scatter_plot.path)

## Build pipeline

In [ ]:
@dsl.pipeline
def mobile_price_classification_pipeline(
    train_data_path: str,
    test_data_path: str,
    test_size: float = 0.5,
    C: List = [1, 0.1, 0.25, 0.5, 2, 0.75],
    kernel: List = ["linear", "rbf"],
    gamma: List = ["auto", 0.01, 0.001, 0.0001, 1],
    decision_function_shape: List[str] = ["ovo", "ovr"],
    scatter_plot_column_x: str = 'ram',
    scatter_plot_column_y: str = 'battery_power',
    seed: int = 42,
):
    """
    Define the mobile price classification pipeline.
    
    This pipeline covers the following steps:
    1. Read data from the specified paths.
    2. Split the data into training and validation sets.
    3. Fit the MinMax scaler.
    4. Tune hyperparameters for the SVM model.
    5. Train the SVM model with the best hyperparameters.
    6. Evaluate the trained model.
    7. Test the model and visualize the results with a scatter plot.
    """
    
    # Step 1: Read the data
    read_data_task = read_data(
        train_data_path=train_data_path,
        test_data_path=test_data_path,
    )
    # Use the cluster internal s3 endpoint
    read_data_task.set_env_variable('AWS_ENDPOINT_URL', f'http://{os.environ["S3_ENDPOINT"]}')
    # Use Kubernetes secrets to provide AWS credentials to the read_data component
    kubernetes.use_secret_as_env(
        read_data_task,
        secret_name='s3creds',
        secret_key_to_env={
            'AWS_ACCESS_KEY_ID': 'AWS_ACCESS_KEY_ID',
            'AWS_SECRET_ACCESS_KEY': 'AWS_SECRET_ACCESS_KEY',
        }
    ) 
   
    # Step 2: Split the data
    split_data_task = split_data(
        train_df=read_data_task.outputs['train_df'],
        test_size=test_size,
        seed=seed
    )

    # Step 3: Fit the scaler
    fit_scaler_task = fit_scaler(
        train_x=split_data_task.outputs['x_train_df']
    )

    # Step 4: Tune hyperparameters
    tune_hyperparams_task = tune_hyperparams(
        train_x=split_data_task.outputs['x_train_df'],
        train_y=split_data_task.outputs['y_train_df'],
        fitted_scaler=fit_scaler_task.outputs['fitted_scaler']
    )

    # Step 5: Train the model
    train_model_task = train_model(
        train_x=split_data_task.outputs['x_train_df'],
        train_y=split_data_task.outputs['y_train_df'],
        hparams=tune_hyperparams_task.output,
        fitted_scaler=fit_scaler_task.outputs['fitted_scaler']
    )

    # Step 6: Evaluate the model
    evaluate_model_task = evaluate_model(
        val_x=split_data_task.outputs['x_val_df'],
        val_y=split_data_task.outputs['y_val_df'],
        trained_model=train_model_task.outputs['trained_model'],
        fitted_scaler=fit_scaler_task.outputs['fitted_scaler'],
        mlflow_run=train_model_task.outputs['Output'],
    )

    # Step 7: Test the model and visualize
    test_model_task = test_model(
        test_x=read_data_task.outputs['test_df'],
        trained_model=train_model_task.outputs['trained_model'],
        fitted_scaler=fit_scaler_task.outputs['fitted_scaler'],
        column_x=scatter_plot_column_x,
        column_y=scatter_plot_column_y
    )

    # Inject MLflow credentials into tasks that need them
    add_env_vars_to_tasks([train_model_task, evaluate_model_task])


## Compile and run pipeline

In [ ]:
# Initialize the Kubeflow Pipelines client
client = Client()

# Define the arguments to be passed to the pipeline
args = dict(
    train_data_path=f"s3://{s3_dataset_path}/train.csv",
    test_data_path=f"s3://{s3_dataset_path}/test.csv",
    test_size=0.2,
    C=[1, 0.1, 0.25, 0.5, 2, 0.75],
    kernel=["linear", "rbf"],
    gamma=["auto", 0.01, 0.001, 0.0001, 1],
    decision_function_shape=["ovo", "ovr"],
    scatter_plot_column_x='ram',
    scatter_plot_column_y='battery_power',
    seed=42
)

# Create a new run from the pipeline function
client.create_run_from_pipeline_func(
    mobile_price_classification_pipeline,
    arguments=args,
    experiment_name="tracked-mobile-price-classification",
    enable_caching=True,
)